# Tamil transcription — quick sample test

Validates that `vasista22/whisper-tamil-large-v2` produces a usable Tamil
transcript on your real call audio before committing GPU hours to 4,000 files.

**Setup:**
1. Settings -> **Accelerator -> GPU** (T4 x2 or single T4 — either works).
2. **+ Add Data** -> upload your `sample/` folder as a private dataset, attach it.
3. Run all cells. The notebook picks a file, trims it to ~4 min, transcribes,
   and prints the transcript.

Expected runtime: ~5-7 min total (model download + conversion ~4 min, decode <1 min).

In [ ]:
!nvidia-smi -L
!pip -q install faster-whisper ctranslate2

In [ ]:
import os, glob, subprocess

MODEL_HF = 'vasista22/whisper-tamil-large-v2'   # production candidate
CT2_DIR = '/kaggle/working/model-ct2'
TRIM_SECONDS = 240                               # 4-min sample
AUDIO_EXTS = ('.mp3', '.wav', '.m4a', '.flac', '.ogg', '.aac', '.opus', '.amr', '.3gp')

# Find every audio file under the attached datasets.
files = [p for p in glob.glob('/kaggle/input/**/*.*', recursive=True)
         if p.lower().endswith(AUDIO_EXTS)]
print('audio files found:', len(files))
for p in files:
    d = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                        '-of', 'csv=p=0', p], capture_output=True, text=True)
    try: dur = float(d.stdout.strip())
    except ValueError: dur = 0
    print(f'  {dur/60:5.1f} min  {p}')

In [ ]:
# Pick the shortest file >= 1 min; trim anything to TRIM_SECONDS so the test
# is fast even if you only uploaded long calls.
def duration(p):
    r = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                        '-of', 'csv=p=0', p], capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except ValueError: return 0.0

src = min(files, key=duration)
SAMPLE = '/kaggle/working/sample.wav'
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', src,
                '-t', str(TRIM_SECONDS), '-ar', '16000', '-ac', '1', SAMPLE], check=True)
print(f'using {duration(src)/60:.1f} min of: {src}')

In [ ]:
# One-time conversion of the HF checkpoint to CTranslate2 int8_float16.
if not os.path.isdir(CT2_DIR):
    !ct2-transformers-converter --model vasista22/whisper-tamil-large-v2 \
        --output_dir /kaggle/working/model-ct2 \
        --copy_files tokenizer.json preprocessor_config.json \
        --quantization int8_float16
else:
    print('model already converted')

In [ ]:
import time
from faster_whisper import WhisperModel

model = WhisperModel(CT2_DIR, device='cuda', compute_type='int8_float16')

t0 = time.time()
segments, info = model.transcribe(
    SAMPLE,
    language='ta',                     # never auto-detect: Tamil sometimes flips to Telugu
    beam_size=5,
    vad_filter=True,                   # skip silence
    vad_parameters={'threshold': 0.35, 'min_silence_duration_ms': 500,
                    'speech_pad_ms': 200},  # relaxed: defaults clip first words
)
lines = []
for s in segments:
    lines.append(f'[{s.start:7.1f} -> {s.end:7.1f}] {s.text}')

elapsed = time.time() - t0
print(f'Transcribed {info.duration:.0f}s in {elapsed:.1f}s  (RTF {elapsed/info.duration:.3f})')
print(f'Detected lang: {info.language} (p={info.language_probability:.2f})')
print('=' * 70)
print('\n'.join(lines))

with open('/kaggle/working/transcript.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

## Reading the result

- **Transcript looks right** -> proceed to the full pipeline
  (`notebooks/kaggle_transcribe.ipynb`).
- **Mostly wrong / wrong script** -> try the code-mix-tuned fallback:
  change `MODEL_HF` to `deepdml/whisper-large-v3-turbo-ta-mix-norm`, delete
  `/kaggle/working/model-ct2`, re-run.
- **Halved words at chunk edges** -> raise `speech_pad_ms` to 300-400.

RTF on T4 should be ~0.02-0.06 with int8_float16. If it is near 1.0 you are
on CPU — check the accelerator setting.